# Approach 6 — report/classifier disagreement review with Grad-CAM

This is an exploratory error-review tool. It compares the coarse image classifier from notebook 06 with finding words extracted from Qwen report drafts, then draws Grad-CAM for the classifier's abnormal-class score. Disagreement is a review flag, not proof that either output is correct. Grad-CAM is not a lesion detector.

## Required files

Run notebook 06 first to create `outputs/06_fusion_classifier/classifier_bundle.joblib`. The generated-report CSV is read from Drive or the local `qwen3vl_cxr` artifact folder. This notebook does not train a new diagnostic model.

In [ ]:
from pathlib import Path
import os, sys
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass
CANDIDATES = [Path.cwd(), Path.cwd().parent, Path("/content/drive/MyDrive/cxr_second_reader"), Path("/content/cxr_second_reader")]
REPO_ROOT = next((p.resolve() for p in CANDIDATES if (p / "src" / "data.py").exists()), None)
if REPO_ROOT is None and Path("/content").exists():
    import subprocess
    clone_dir = Path("/content/cxr_second_reader")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "https://github.com/eyadstein/cxr_second_reader.git", str(clone_dir)], check=True)
    if (clone_dir / "src" / "data.py").exists():
        REPO_ROOT = clone_dir.resolve()
if REPO_ROOT is None:
    raise FileNotFoundError("Open this notebook from the cxr_second_reader project checkout, or make the updated project available in Drive/GitHub.")
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
print("Project:", REPO_ROOT)

In [ ]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import torch
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import transforms
import matplotlib.pyplot as plt
from src.experiment_utils import study_rows
from src.data import resolve_image
from src.evaluate import extract_labels

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BUNDLE_PATH = Path("outputs/06_fusion_classifier/classifier_bundle.joblib")
if not BUNDLE_PATH.is_file():
    raise FileNotFoundError("Run notebook 06 first to create the image classifier bundle.")
bundle = joblib.load(BUNDLE_PATH)
image_clf, scaler = bundle["image_classifier"], bundle["image_scaler"]
print("device:", DEVICE)

In [ ]:
RESULTS_CANDIDATES = [Path("/content/drive/MyDrive/qwen3vl_cxr"), REPO_ROOT / "qwen3vl_cxr"]
RESULTS = next((p for p in RESULTS_CANDIDATES if (p / "eval_qwen_lora_finetuned.csv").is_file()), None)
if RESULTS is None:
    raise FileNotFoundError("No eval_qwen_lora_finetuned.csv found in Drive or project artifacts.")
predictions = pd.read_csv(RESULTS / "eval_qwen_lora_finetuned.csv")
test_df = study_rows("test", None)
predictions["uid"] = predictions["uid"].astype(str)
test_df["uid"] = test_df["uid"].astype(str)
joined = predictions.merge(test_df[["uid", "class"]], on="uid", how="inner", validate="one_to_one")

def report_says_abnormal(report):
    labels = extract_labels(report)
    return int(labels[1:].sum() > 0)

joined["report_abnormal_claim"] = joined["prediction"].fillna("").map(report_says_abnormal)
print("Rows matched to held-out metadata:", len(joined))
print("These are candidate disagreements only; the report heuristic and classifier may both be wrong.")

In [ ]:
# Load the matching pretrained ResNet-18 feature layers for Grad-CAM.
weights = ResNet18_Weights.DEFAULT
resnet = resnet18(weights=weights).to(DEVICE).eval()
for parameter in resnet.parameters(): parameter.requires_grad = False
feature_layers = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
                               resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4).to(DEVICE)
preprocess = transforms.Compose([transforms.Resize((224,224)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])])
coef = image_clf.coef_[0] / scaler.scale_
coef_t = torch.tensor(coef, dtype=torch.float32, device=DEVICE)

def show_gradcam(row):
    path = resolve_image(row)
    from PIL import Image
    with Image.open(path) as source:
        original = source.convert("RGB").copy()
    x = preprocess(original).unsqueeze(0).to(DEVICE).requires_grad_(True)
    fmap = feature_layers(x)
    pooled = torch.nn.functional.adaptive_avg_pool2d(fmap, 1).flatten(1)
    abnormal_logit = (pooled[0] * coef_t).sum()
    gradient = torch.autograd.grad(abnormal_logit, fmap)[0]
    weights_cam = gradient.mean(dim=(2,3), keepdim=True)
    cam = torch.relu((weights_cam * fmap).sum(dim=1, keepdim=True))
    cam = torch.nn.functional.interpolate(cam, size=(224,224), mode="bilinear", align_corners=False)[0,0]
    cam = cam.detach().cpu().numpy(); cam = cam / (cam.max() + 1e-8)
    probabilities = image_clf.predict_proba(scaler.transform(pooled.detach().cpu().numpy()))[0]
    fig, ax = plt.subplots(1,2,figsize=(10,4))
    ax[0].imshow(original, cmap="gray"); ax[0].set_title("X-ray (for review)"); ax[0].axis("off")
    ax[1].imshow(original.resize((224,224)), cmap="gray"); ax[1].imshow(cam, cmap="jet", alpha=0.35)
    ax[1].set_title(f"Grad-CAM, abnormal score; p={probabilities[1]:.2f}"); ax[1].axis("off")
    plt.show()
    return float(probabilities[1])

from PIL import Image

@torch.inference_mode()
def image_abnormal_probability(row):
    with Image.open(resolve_image(row)) as source:
        image = source.convert("RGB").copy()
    x = preprocess(image).unsqueeze(0).to(DEVICE)
    features = feature_layers(x)
    pooled = torch.nn.functional.adaptive_avg_pool2d(features, 1).flatten(1).cpu().numpy()
    return float(image_clf.predict_proba(scaler.transform(pooled))[0, 1])

if len(joined):
    joined["image_abnormal_probability"] = [
        image_abnormal_probability(row) for _, row in joined.iterrows()
    ]
    joined["disagreement_candidate"] = (
        joined["report_abnormal_claim"] != (joined["image_abnormal_probability"] >= 0.5).astype(int)
    )
    display(joined.loc[joined["disagreement_candidate"], ["uid", "prediction", "report_abnormal_claim", "image_abnormal_probability"]].head(10))
    show_gradcam(joined.iloc[0])
else:
    print("No matching report rows; check whether this model's evaluation IDs are in the current test split.")

## Interpretation

The image classifier predicts only the dataset's normal/non-normal class; it was trained with weak labels and ImageNet features. The report parser uses keywords and negation heuristics. Neither is reliable enough to adjudicate findings. Review the original X-ray and reference report for every candidate; do not use the heatmap as a localization or diagnostic result.